In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp04/ex_5.py ──
"""
Shared infrastructure for MLFP04 Exercise 5 — Association Rules.

Contains: synthetic Singapore retail transaction generator, category map,
rule dataclass helpers, and small polars utilities used by every technique
file in ``modules/mlfp04/solutions/ex_5/``.

Technique-specific code (Apriori from scratch, FP-Growth wrapper, rule
evaluation, feature engineering for classification) does NOT belong here —
it lives in the per-technique files.
"""

import asyncio
import re
from collections import defaultdict
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import polars as pl

from kailash_ml import ExperimentTracker


setup_environment()

# ════════════════════════════════════════════════════════════════════════
# OUTPUT DIRECTORY
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "mlfp04_ex5_association_rules"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# SINGAPORE RETAIL PRODUCT CATALOGUE
# ════════════════════════════════════════════════════════════════════════
# 25 products grouped to mirror a typical HDB neighbourhood mini-mart basket.

PRODUCTS: list[str] = [
    "bread",
    "butter",
    "milk",
    "eggs",
    "rice",
    "noodles",
    "soy_sauce",
    "cooking_oil",
    "chicken",
    "fish",
    "coffee",
    "tea",
    "sugar",
    "condensed_milk",
    "biscuits",
    "chips",
    "soft_drink",
    "beer",
    "wine",
    "tissue",
    "shampoo",
    "soap",
    "detergent",
    "toothpaste",
    "bananas",
]

CATEGORY_MAP: dict[str, str] = {
    "bread": "breakfast",
    "butter": "breakfast",
    "eggs": "breakfast",
    "milk": "dairy",
    "condensed_milk": "dairy",
    "coffee": "beverage",
    "tea": "beverage",
    "soft_drink": "beverage",
    "sugar": "pantry",
    "rice": "pantry",
    "cooking_oil": "pantry",
    "soy_sauce": "pantry",
    "noodles": "pantry",
    "chicken": "protein",
    "fish": "protein",
    "beer": "alcohol",
    "wine": "alcohol",
    "chips": "snack",
    "biscuits": "snack",
    "bananas": "fruit",
    "shampoo": "personal_care",
    "soap": "personal_care",
    "toothpaste": "personal_care",
    "tissue": "household",
    "detergent": "household",
}

# Co-purchase bundles (items, probability). Models real behaviour:
# kaya-toast breakfast, kopi-C, household replenishment, beer+chips, etc.
BUNDLES: list[tuple[list[str], float]] = [
    (["bread", "butter", "eggs"], 0.15),
    (["coffee", "condensed_milk", "sugar"], 0.12),
    (["rice", "chicken", "soy_sauce"], 0.10),
    (["noodles", "eggs", "soy_sauce"], 0.08),
    (["beer", "chips"], 0.09),
    (["milk", "biscuits"], 0.07),
    (["shampoo", "soap", "toothpaste"], 0.06),
    (["tea", "sugar", "biscuits"], 0.05),
    (["wine", "chips", "biscuits"], 0.04),
    (["cooking_oil", "rice", "fish"], 0.06),
    (["detergent", "tissue", "soap"], 0.05),
    (["bananas", "milk", "eggs"], 0.05),
]

N_TRANSACTIONS_DEFAULT: int = 2500


# ════════════════════════════════════════════════════════════════════════
# TRANSACTION GENERATOR
# ════════════════════════════════════════════════════════════════════════


def generate_transactions(
    n: int = N_TRANSACTIONS_DEFAULT,
    seed: int = 42,
) -> list[set[str]]:
    """Generate synthetic Singapore retail transactions.

    Each transaction is a set of product strings. Bundles fire with their
    listed probability; each item inside a firing bundle is kept with 0.85
    probability (random drop-out) so support is noisy. A Poisson number of
    random items is added on top to simulate impulse buys.
    """
    rng = np.random.default_rng(seed)
    transactions: list[set[str]] = []
    for _ in range(n):
        basket: set[str] = set()
        for bundle_items, prob in BUNDLES:
            if rng.random() < prob:
                for item in bundle_items:
                    if rng.random() < 0.85:
                        basket.add(item)
        n_random = rng.poisson(2)
        if n_random > 0:
            random_items = rng.choice(
                PRODUCTS, size=int(min(n_random, 5)), replace=False
            )
            basket.update(random_items)
        if basket:
            transactions.append(basket)
    return transactions


def transactions_to_onehot(transactions: list[set[str]]) -> pl.DataFrame:
    """One-row-per-transaction boolean matrix (columns = sorted PRODUCTS).

    Polars-native. Used as input to mlxtend FP-Growth (via .to_pandas()).
    """
    all_items = sorted(PRODUCTS)
    rows = [{item: (item in txn) for item in all_items} for txn in transactions]
    return pl.DataFrame(rows)


def product_frequency(transactions: Iterable[set[str]]) -> dict[str, int]:
    """Count how many transactions contain each product."""
    counts: dict[str, int] = defaultdict(int)
    for txn in transactions:
        for item in txn:
            counts[item] += 1
    return dict(counts)


def print_transaction_summary(transactions: list[set[str]]) -> None:
    """One-liner summary + top 10 product frequency. Used by every file."""
    avg_basket = float(np.mean([len(t) for t in transactions]))
    print("=== Synthetic Singapore Retail Transactions ===")
    print(f"  Transactions: {len(transactions):,}")
    print(f"  Products:     {len(PRODUCTS)}")
    print(f"  Avg basket:   {avg_basket:.1f} items")

    freq = product_frequency(transactions)
    n = len(transactions)
    print("\n  Top 10 products by frequency:")
    for item, count in sorted(freq.items(), key=lambda kv: -kv[1])[:10]:
        print(f"    {item:<20} {count:>5} ({count / n:.1%})")


# ════════════════════════════════════════════════════════════════════════
# RULE HELPERS
# ════════════════════════════════════════════════════════════════════════


def format_itemset(items: Iterable[str]) -> str:
    """Deterministic pretty-print of a frozenset of items."""
    return ", ".join(sorted(items))


def categorise_rule(
    antecedent: frozenset[str],
    consequent: frozenset[str],
) -> tuple[set[str], set[str], str]:
    """Return (antecedent_categories, consequent_categories, relation_type)."""
    ant_cats = {CATEGORY_MAP.get(item, "other") for item in antecedent}
    con_cats = {CATEGORY_MAP.get(item, "other") for item in consequent}
    if ant_cats == con_cats:
        rel = "within-category complement"
    elif ant_cats & con_cats:
        rel = "cross-category with overlap"
    else:
        rel = "cross-category association"
    return ant_cats, con_cats, rel


def rules_to_polars(rules: list[dict]) -> pl.DataFrame:
    """Convert a list of rule dicts into a polars DataFrame for plotting."""
    return pl.DataFrame(
        {
            "antecedent": [format_itemset(r["antecedent"]) for r in rules],
            "consequent": [format_itemset(r["consequent"]) for r in rules],
            "support": [float(r["support"]) for r in rules],
            "confidence": [float(r["confidence"]) for r in rules],
            "lift": [float(r["lift"]) for r in rules],
        }
    )


# ════════════════════════════════════════════════════════════════════════
# KAILASH-ML EXPERIMENT TRACKER — shared by every association-rules technique
# ════════════════════════════════════════════════════════════════════════
# Every M4 ex_5 lesson logs its mining + rule-evaluation + classifier runs
# into a single SQLite store so students can compare techniques side by side
# (Apriori vs FP-Growth runtime, rule-quality at varying thresholds,
# baseline vs rule-enhanced classifier metrics) after the lesson group ends.
# Mirrors m4_clustering_zoo / m4_dimreduction_zoo / m4_anomaly_zoo from
# ex_1 / ex_3 / ex_4.
#
# IMPORTANT: this store is SEPARATE from the other M4 stores. Per the
# SQLite contention trap (.session-notes), running two ExperimentTracker
# writers against the same SQLite file concurrently triggers `disk I/O
# error`. Each exercise group gets its own DB.

ASSOC_DB = "sqlite:///mlfp04_ex5_assoc_rules.db"
ASSOC_EXPERIMENT_NAME = "m4_assoc_rules_zoo"

_TRACKER_KEY_RE = re.compile(r"[^a-zA-Z0-9_.\-]")


def _slug(text: str) -> str:
    """Coerce free-form prose into a tracker-key-safe slug.

    ExperimentTracker enforces ``^[a-zA-Z_][a-zA-Z0-9_.\\-]*$`` on metric
    names. Replace forbidden characters with underscores; ensure a leading
    alpha character; collapse repeats.
    """
    s = _TRACKER_KEY_RE.sub("_", text.strip())
    s = re.sub(r"_+", "_", s).strip("_") or "metric"
    if not s[0].isalpha() and s[0] != "_":
        s = f"m_{s}"
    return s


def _finite(x: float) -> float:
    """Tracker rejects NaN/inf via MetricValueError; coerce to 0.0.

    Association-rule metrics (lift, conviction) can collapse to inf when
    confidence saturates at 1.0; aggregate scalars over empty filter sets
    can NaN. Every emit through this guard.
    """
    return float(x) if x == x and x not in (float("inf"), float("-inf")) else 0.0


async def _setup_engines_async() -> tuple[ExperimentTracker, str]:
    """Open the association-rules ExperimentTracker (kailash-ml ≥1.5)."""
    tracker = await ExperimentTracker.create(store_url=ASSOC_DB)
    return tracker, ASSOC_EXPERIMENT_NAME


def setup_engines() -> tuple[ExperimentTracker, str]:
    """Sync wrapper. Returns (tracker, experiment_name)."""
    return asyncio.run(_setup_engines_async())


def teardown_engines(tracker: ExperimentTracker) -> None:
    """Drain the aiosqlite worker threads before the script returns.

    kailash's AsyncSQLitePool spawns NON-DAEMON aiosqlite worker threads on
    first pool use. Python 3.13's ``Py_FinalizeEx`` joins non-daemon threads
    BEFORE running ``atexit`` handlers, so an atexit-based close runs too
    late — the interpreter hangs forever in ``wait_for_thread_shutdown``
    waiting on workers stuck in ``queue.get()``.

    Solutions MUST call ``teardown_engines(tracker)`` after the REFLECTION
    block (or use it via ``try: … finally: teardown_engines(tracker)``).
    See ``rules/patterns.md`` § "Async Resource Cleanup": real cleanup is
    the caller's responsibility; ``close()`` must NOT live in ``__del__``.
    """
    asyncio.run(tracker.close())


async def _track_run_async(
    tracker: ExperimentTracker,
    exp_name: str,
    run_name: str,
    params: dict[str, Any],
    scalar_metrics: dict[str, float],
    series_metrics: dict[str, list[float]] | None = None,
) -> None:
    """Log one lesson's run: scalar metrics + optional per-step series."""
    async with tracker.track(experiment=exp_name, run_name=run_name) as run:
        await run.log_params({k: str(v) for k, v in params.items()})
        for name, value in scalar_metrics.items():
            await run.log_metric(_slug(name), _finite(value))
        if series_metrics:
            for name, values in series_metrics.items():
                slug = _slug(name)
                for step, value in enumerate(values, start=1):
                    await run.log_metric(slug, _finite(value), step=step)


def track_run(
    tracker: ExperimentTracker,
    exp_name: str,
    run_name: str,
    params: dict[str, Any],
    scalar_metrics: dict[str, float],
    series_metrics: dict[str, list[float]] | None = None,
) -> None:
    """Sync wrapper for logging a single technique's run."""
    asyncio.run(
        _track_run_async(
            tracker, exp_name, run_name, params, scalar_metrics, series_metrics
        )
    )




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 5.4: Rule-Based Features for Supervised Classification
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Engineer features from discovered association rules
#   - Compare a product-presence baseline against a rule-enhanced model
#   - Attribute feature importance across product vs rule groups
#
# PREREQUISITES:
#   - 01_apriori_from_scratch.py
#   - 03_rule_evaluation.py
#   - MLFP03 Exercise 1 (feature engineering)
#
# ESTIMATED TIME: ~40 min
#
# TASKS:
#   1. Theory — rules as handcrafted co-occurrence features
#   2. Build — turn actionable rules into numeric feature columns
#   3. Train — LR + RF, baseline vs rules-only vs combined
#   4. Visualise — metric comparison + feature importance attribution
#   5. Apply — NTUC Link high-value shopper scoring
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

from collections import defaultdict
from itertools import combinations

import numpy as np
import polars as pl
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler



### Kailash-ML ExperimentTracker — association-rules zoo shared store


In [ ]:
tracker, exp_name = setup_engines()



## THEORY — Rules as Handcrafted Co-Occurrence Features


In [ ]:
# A linear model on raw one-hot product features sees each product as
# independent. Association rules inject interaction structure that the
# linear model cannot represent on its own.
#
# The forward connection:
#   Manual rules  ->  Linear factorisation (Ex 7)  ->  Neural nets (Ex 8)
# Each step replaces more of the human steering with learned structure.



## MINI-APRIORI + RULE GENERATION (inline, file stands alone)


In [ ]:
def _apriori(
    transactions: list[set[str]], min_support: float
) -> dict[frozenset[str], float]:
    n = len(transactions)
    min_count = min_support * n
    item_counts: dict[str, int] = defaultdict(int)
    for txn in transactions:
        for item in txn:
            item_counts[item] += 1
    freq: dict[frozenset[str], float] = {}
    level: list[frozenset[str]] = []
    for item, count in item_counts.items():
        if count >= min_count:
            fs = frozenset([item])
            freq[fs] = count / n
            level.append(fs)
    k = 2
    while level:
        prev_set = set(level)
        candidates: set[frozenset[str]] = set()
        for i, a in enumerate(level):
            for b in level[i + 1 :]:
                u = a | b
                if len(u) == k and all((u - frozenset([it])) in prev_set for it in u):
                    candidates.add(u)
        if not candidates:
            break
        counts: dict[frozenset[str], int] = defaultdict(int)
        for txn in transactions:
            tf = frozenset(txn)
            for c in candidates:
                if c.issubset(tf):
                    counts[c] += 1
        level = []
        for c, ct in counts.items():
            if ct >= min_count:
                freq[c] = ct / n
                level.append(c)
        k += 1
    return freq


def _rules_from_itemsets(
    freq: dict[frozenset[str], float],
    min_confidence: float = 0.4,
    min_lift: float = 1.5,
) -> list[dict]:
    rules: list[dict] = []
    for itemset, support in freq.items():
        if len(itemset) < 2:
            continue
        items = list(itemset)
        for r in range(1, len(items)):
            for ant_tuple in combinations(items, r):
                antecedent = frozenset(ant_tuple)
                consequent = itemset - antecedent
                supp_ant = freq.get(antecedent)
                supp_con = freq.get(consequent)
                if supp_ant is None or supp_con is None:
                    continue
                confidence = support / supp_ant
                lift = confidence / supp_con
                if confidence >= min_confidence and lift > min_lift:
                    rules.append(
                        {
                            "antecedent": antecedent,
                            "consequent": consequent,
                            "support": support,
                            "confidence": confidence,
                            "lift": lift,
                        }
                    )
    rules.sort(key=lambda r: -r["lift"])
    return rules



## TASK 2 — BUILD: rule-based feature engineer


Columns produced per rule:
      - rule{i}_antecedent — 1 if antecedent present
      - rule{i}_full       — 1 if antecedent AND consequent present

    Plus aggregates: n_rules_triggered, total_rule_lift_x100,
    max_rule_lift_x100.


In [ ]:
def engineer_rule_features(
    transactions: list[set[str]],
    rules: list[dict],
) -> pl.DataFrame:
    rows: list[dict[str, int]] = []
    for txn in transactions:
        txn_set = frozenset(txn)
        row: dict[str, int] = {}
        total_lift = 0.0
        n_triggered = 0
        max_lift = 0.0

        for idx, rule in enumerate(rules):
            # TODO: compute ant_present (is the antecedent a subset of txn?)
            # and full_present (is antecedent UNION consequent a subset?).
            # Hint: frozenset.issubset(txn_set); union with the | operator.
            ant_present = ____
            full_present = ____

            row[f"rule{idx}_antecedent"] = ant_present
            row[f"rule{idx}_full"] = full_present

            if full_present:
                n_triggered += 1
                total_lift += float(rule["lift"])
                if float(rule["lift"]) > max_lift:
                    max_lift = float(rule["lift"])

        row["n_rules_triggered"] = n_triggered
        row["total_rule_lift_x100"] = int(total_lift * 100)
        row["max_rule_lift_x100"] = int(max_lift * 100)
        rows.append(row)

    return pl.DataFrame(rows).fill_null(0)



## TASK 3 — TRAIN: baseline vs rules-only vs combined


In [ ]:
transactions = generate_transactions(n=2500, seed=42)
print_transaction_summary(transactions)

basket_sizes = np.array([len(t) for t in transactions])
HIGH_VALUE_THRESHOLD = 6
y = (basket_sizes >= HIGH_VALUE_THRESHOLD).astype(int)
print(f"\n  Target: high-value shopper (basket >= {HIGH_VALUE_THRESHOLD} items)")
print(f"  Positive rate: {y.mean():.1%}")

# Baseline features
onehot = transactions_to_onehot(transactions)
X_baseline = onehot.to_numpy().astype(np.float64)
print(f"\n  Baseline features (product presence): {X_baseline.shape[1]}")

# Mine rules and engineer rule features
print("\n=== Mining rules for feature engineering ===")
freq = _apriori(transactions, min_support=0.03)
rules = _rules_from_itemsets(freq, min_confidence=0.4, min_lift=1.5)
top_rules = rules[:20]
print(f"  Actionable rules found: {len(rules)}")
print(f"  Using top {len(top_rules)} rules as features")

rule_df = engineer_rule_features(transactions, top_rules)
X_rules = rule_df.to_numpy().astype(np.float64)

# TODO: stack baseline + rule features horizontally into X_combined.
# Hint: np.hstack([...])
X_combined = ____

print(f"  Rule features:     {X_rules.shape[1]}")
print(f"  Combined features: {X_combined.shape[1]}")


def _split_and_scale(X: np.ndarray, y: np.ndarray, scale: bool):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    if scale:
        s = StandardScaler()
        X_tr = s.fit_transform(X_tr)
        X_te = s.transform(X_te)
    return X_tr, X_te, y_tr, y_te


results: dict[str, dict[str, float]] = {}

print("\n=== Model: Logistic Regression ===")
for name, X in [
    ("Baseline", X_baseline),
    ("Rules only", X_rules),
    ("Combined", X_combined),
]:
    X_tr, X_te, y_tr, y_te = _split_and_scale(X, y, scale=True)
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_tr, y_tr)
    y_pred = lr.predict(X_te)
    y_proba = lr.predict_proba(X_te)[:, 1]
    acc = accuracy_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred)
    auc = roc_auc_score(y_te, y_proba)
    results[f"LR: {name}"] = {"accuracy": acc, "f1": f1, "auc_roc": auc}
    print(f"  {name:<12} acc={acc:.4f} f1={f1:.4f} auc={auc:.4f}")

print("\n=== Model: Random Forest ===")
for name, X in [
    ("Baseline", X_baseline),
    ("Rules only", X_rules),
    ("Combined", X_combined),
]:
    X_tr, X_te, y_tr, y_te = _split_and_scale(X, y, scale=False)
    rf = RandomForestClassifier(
        n_estimators=200, max_depth=10, random_state=42, n_jobs=-1
    )
    rf.fit(X_tr, y_tr)
    y_pred = rf.predict(X_te)
    y_proba = rf.predict_proba(X_te)[:, 1]
    acc = accuracy_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred)
    auc = roc_auc_score(y_te, y_proba)
    results[f"RF: {name}"] = {"accuracy": acc, "f1": f1, "auc_roc": auc}
    print(f"  {name:<12} acc={acc:.4f} f1={f1:.4f} auc={auc:.4f}")



### Checkpoint


In [ ]:
assert (
    X_combined.shape[1] > X_baseline.shape[1]
), "Combined should add columns to the baseline, not replace them"
assert X_rules.shape[1] > 0, "Should have produced at least one rule feature"
lr_baseline = results["LR: Baseline"]["auc_roc"]
lr_combined = results["LR: Combined"]["auc_roc"]
rf_baseline = results["RF: Baseline"]["auc_roc"]
assert lr_baseline > 0.5, "Baseline LR should beat random"
assert rf_baseline > 0.5, "Baseline RF should beat random"
assert (
    lr_combined >= lr_baseline - 0.05
), "Adding rule features should not significantly regress LR"
print("\n[ok] Checkpoint passed — rule-enhanced model trained + compared\n")



## TASK 4 — VISUALISE: feature importance attribution


In [ ]:
X_tr, X_te, y_tr, y_te = _split_and_scale(X_combined, y, scale=False)
rf_combined = RandomForestClassifier(
    n_estimators=200, max_depth=10, random_state=42, n_jobs=-1
)
rf_combined.fit(X_tr, y_tr)
importances = rf_combined.feature_importances_

# TODO: split the RF importance vector into the product-feature sum and
# the rule-feature sum. The first X_baseline.shape[1] entries are
# product features; everything after is rule features.
product_importance = ____
rule_importance = ____
total = product_importance + rule_importance

print("=== Feature Importance Attribution (RF combined) ===")
print(f"  Product features contribute: {product_importance / total:.1%}")
print(f"  Rule features contribute:    {rule_importance / total:.1%}")

all_feature_names = list(onehot.columns) + list(rule_df.columns)
top_idx = np.argsort(importances)[::-1][:15]
print("\n  Top 15 features in the combined model:")
for idx in top_idx:
    fname = all_feature_names[idx]
    ftype = "product" if idx < X_baseline.shape[1] else "rule"
    print(f"    [{ftype:>7}] {fname:<40} {importances[idx]:.4f}")

metric_rows = [{"model": k, **v} for k, v in results.items()]
pl.DataFrame(metric_rows).write_csv(OUTPUT_DIR / "rule_features_metrics.csv")



## TASK 5 — APPLY: NTUC Link high-value shopper scoring


In [ ]:
# SCENARIO: NTUC Link (~1.7M members) needs an auditable, reproducible
# model to flag the top 20% of baskets as high-value for targeted
# next-trip offers. Rule-based features give the CRM team the per-feature
# interpretability that pure matrix factorisation would hide.
#
# BUSINESS IMPACT: 3-5% precision improvement at the top 20% cutoff
# reaches 25K-42K extra members/month; S$4-6 offer ROI per member
# ~= S$100K-250K/month in incremental campaign contribution.



## TRACK — Log baseline-vs-rule-enhanced classifier metrics


In [ ]:
# Same shared experiment as 01-03. Logs all four model variants side by
# side (LR Baseline / LR Combined / RF Baseline / RF Combined) plus the
# product-vs-rule importance split. Series = top-15 RF importances.

# TODO: pick run_name="rule_features_vs_baseline" and fill in the two
# AUC-lift scalars (combined - baseline) for LR and RF.
lr_combined_lift = ____
rf_combined_lift = ____

track_run(
    tracker,
    exp_name,
    run_name=____,
    params={
        "algorithm": "rule_features",
        "implementation": "lr_plus_rf_baseline_vs_combined",
        "n_transactions": len(transactions),
        "n_baseline_features": int(X_baseline.shape[1]),
        "n_rule_features": int(X_rules.shape[1]),
        "n_combined_features": int(X_combined.shape[1]),
        "rf_n_estimators": 200,
        "rf_max_depth": 10,
    },
    scalar_metrics={
        "lr_baseline_auc": float(results["LR: Baseline"]["auc_roc"]),
        "lr_combined_auc": float(results["LR: Combined"]["auc_roc"]),
        "lr_auc_lift_combined_vs_baseline": float(lr_combined_lift),
        "rf_baseline_auc": float(results["RF: Baseline"]["auc_roc"]),
        "rf_combined_auc": float(results["RF: Combined"]["auc_roc"]),
        "rf_auc_lift_combined_vs_baseline": float(rf_combined_lift),
        "product_importance_share": float(product_importance / total),
        "rule_importance_share": float(rule_importance / total),
    },
    series_metrics={
        "rf_top15_importances": [float(importances[i]) for i in top_idx],
    },
)
print(f"  [tracked] Baseline-vs-combined classifier metrics logged to {exp_name}\n")



## DESTINATION-FIRST CLOSE — kailash-ml FeatureEngineer + TrainingPipeline


In [ ]:
# The hand-built rule-feature columns made every interaction explicit
# (auditability win). The Kailash destination stacks two engines: ship
# the rule columns alongside auto-discovered co-occurrence features from
# FeatureEngineer, then let TrainingPipeline train + rank LR vs RF vs
# gradient-boosted variants in one call. Manual rules become one feature
# group among many — the bottom rung of the discovery spectrum.

print("  Destination contract:")
print("    FeatureEngineer().generate(df)             -> auto co-occurrence cols")
print("    TrainingPipeline().train(schema, model, eval) -> ranked classifiers")
print(
    f"  Today's run: {X_rules.shape[1]} hand-built rule features + "
    f"4 sklearn variants — top AUC = {max(v['auc_roc'] for v in results.values()):.4f}"
)



## REFLECTION


[x] Engineered numeric features from discovered association rules
  [x] Compared baseline vs rules-only vs combined models
  [x] Attributed feature importance across product and rule groups
  [x] Pointed at the Kailash destination — FeatureEngineer +
      TrainingPipeline — that ships rule features alongside auto-
      discovered co-occurrence structure in one call

  KEY INSIGHT: Manual rules are the MANUAL end of a spectrum that runs
  all the way to deep learning.
    Manual rules  ->  Linear factorisation (Ex 7)  ->  Neural nets (Ex 8)

  Next: Exercise 6 moves to UNSTRUCTURED text — TF-IDF from scratch,
  NMF topic modelling, and NPMI coherence.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

# Drain the aiosqlite worker threads so Py_Finalize doesn't hang.
teardown_engines(tracker)

